# Comparación de la semana · obstáculos y `reward_slices`

Esta semana corrimos seis experimentos, todos de 500.000 pasos con semilla 0:

| Vista | 2 obstáculos verdes (semana del 24) | 3 obstáculos de colores (semana del 25) |
|---|---|---|
| 5×5 | sin `reward_slices` · con `reward_slices` | con `reward_slices` |
| 3×3 | sin `reward_slices` · con `reward_slices` | con `reward_slices` |

- **2 obstáculos verdes:** un cuadrado y un triángulo, los dos verdes, uno en cada habitación lateral.
- **3 obstáculos de colores:** un cuadrado verde y un triángulo morado en las habitaciones laterales, y un rombo amarillo en la habitación del botón.
- En los dos casos, los obstáculos cambian de casilla al comenzar cada partida y se quedan quietos durante ella.
- **`reward_slices=1`:** de las 16 secuencias de cada batch, una se toma alrededor de un momento en que el agente comió una galleta (si ya comió alguna).

El notebook responde dos preguntas, cada una con un gráfico:

1. **¿Ayuda `reward_slices`?** Compara los obstáculos verdes sin y con `reward_slices`.
2. **¿Importa cómo son los obstáculos?** Compara los verdes con los de colores, todos con `reward_slices`. Como referencia se agrega el run sin obstáculos de la semana del 15.

Al final están los videos de los seis experimentos.

**Usa CPU y ejecuta todo** (Entorno de ejecución → Ejecutar todo). Si falta algún video, la sección 5 lo crea y lo guarda en Drive; la primera vez tarda unos minutos.

## 1. Montar Drive

Usa la cuenta donde están los entrenamientos.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Leer los runs

Lee las métricas de cada run. Si un entrenamiento se retomó desde un checkpoint, se descarta lo que quedó repetido.

In [ ]:
import json
from pathlib import Path

RAIZ = Path("/content/drive/MyDrive/galleta")

# Carpeta en Drive de cada run, por vista y condición.
RUNS = {
    ("5×5", "verdes, sin reward_slices"): "semana24/obstaculos_aleatorios5x5/distinct_s0",
    ("5×5", "verdes, con reward_slices"): "semana24/obstaculos_aleatorios5x5/distinct_sampler1_s0",
    ("5×5", "colores, con reward_slices"): "semana25/tres_obstaculos_colores5x5/distinct_sampler1_s0",
    ("5×5", "sin obstáculos, con reward_slices"): "semana15/cookie_partial_hall5_view5_s0_base_sampler1",
    ("3×3", "verdes, sin reward_slices"): "semana24/obstaculos_aleatorios3x3/distinct_s0",
    ("3×3", "verdes, con reward_slices"): "semana24/obstaculos_aleatorios3x3/distinct_sampler1_s0",
    ("3×3", "colores, con reward_slices"): "semana25/tres_obstaculos_colores3x3/distinct_sampler1_s0",
}
# Cada condición tiene siempre el mismo color, en todos los gráficos.
COLORES = {
    "verdes, sin reward_slices": "#00b894",
    "verdes, con reward_slices": "#257de0",
    "colores, con reward_slices": "#f47230",
    "sin obstáculos, con reward_slices": "#8a8a8a",
}


def leer(carpeta):
    """Evaluaciones y galletas por partida de entrenamiento, como listas de (paso, valor)."""
    evaluaciones, partidas = [], []
    with open(RAIZ / carpeta / "metrics.jsonl") as archivo:
        for linea in archivo:
            fila = json.loads(linea)
            for clave, serie in (("episode/eval_score", evaluaciones), ("episode/score", partidas)):
                if clave in fila:
                    # Al retomar se vuelve a un paso anterior: se borra lo que se repite.
                    while serie and serie[-1][0] >= fila["step"]:
                        serie.pop()
                    serie.append((fila["step"], fila[clave]))
    return evaluaciones, partidas


DATOS = {}
for (vista, condicion), carpeta in RUNS.items():
    try:
        DATOS[vista, condicion] = leer(carpeta)
        evaluaciones, partidas = DATOS[vista, condicion]
        pasos = max(evaluaciones[-1][0], partidas[-1][0])
        print(f"{vista} · {condicion}: hasta {pasos:,} pasos".replace(f"{pasos:,}", f"{pasos:,}".replace(",", ".")))
    except FileNotFoundError:
        print(f"{vista} · {condicion}: NO ENCONTRADO en {carpeta}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

TOPE = 1.08 * max(valor for evaluaciones, _ in DATOS.values() for _, valor in evaluaciones)


def graficar(titulo, lineas):
    """Evaluación del actor en 5×5 y 3×3. lineas: lista de (condición, nombre en la leyenda)."""
    figura, ejes = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True, layout="constrained")
    for eje, vista in zip(ejes, ("5×5", "3×3")):
        faltan = []
        for condicion, nombre in lineas:
            if (vista, condicion) not in DATOS:
                faltan.append(nombre)
                continue
            pasos, valores = np.array(DATOS[vista, condicion][0]).T
            referencia = condicion.startswith("sin obstáculos")
            eje.plot(pasos / 1000, valores, color=COLORES[condicion], linewidth=2,
                     linestyle="--" if referencia else "-", marker="o", markersize=4,
                     zorder=3 if referencia else 2, label=nombre)
        if faltan:
            eje.text(0.02, 0.96, f"No hay run en {vista}: " + ", ".join(faltan).lower(), transform=eje.transAxes,
                     va="top", fontsize=9, color="#666666")
        eje.set_title(f"Vista {vista}", loc="left", fontsize=12)
        eje.set_xlim(0, 500)
        eje.set_ylim(-0.5, TOPE)
        eje.set_xlabel("Pasos de entrenamiento (miles)")
        eje.grid(axis="y", color="#e6e6e6")
        eje.set_axisbelow(True)
        eje.spines[["top", "right"]].set_visible(False)
        eje.spines[["left", "bottom"]].set_color("#999999")
        eje.tick_params(colors="#555555")
    ejes[0].set_ylabel("Galletas por partida\nen la evaluación")
    etiquetas = {}
    for eje in ejes:
        for linea, nombre in zip(*eje.get_legend_handles_labels()):
            etiquetas.setdefault(nombre, linea)
    orden = [nombre for _, nombre in lineas if nombre in etiquetas]
    figura.legend([etiquetas[n] for n in orden], orden, loc="outside lower center",
                  ncol=len(orden), frameon=False, fontsize=11)
    figura.suptitle(titulo, fontsize=14, fontweight="bold")
    plt.show()


def tabla(lineas):
    filas = []
    for vista in ("5×5", "3×3"):
        for condicion, nombre in lineas:
            if (vista, condicion) not in DATOS:
                continue
            evaluaciones, partidas = DATOS[vista, condicion]
            # La evaluación del paso 0 es antes de entrenar: no cuenta.
            primera = next((paso for paso, valor in evaluaciones if paso > 0 and valor > 0), None)
            filas.append({
                "Vista": vista,
                "Run": nombre,
                "Galletas comidas al entrenar": int(sum(valor for _, valor in partidas)),
                "Primera evaluación con galletas": f"{primera // 1000}k" if primera is not None else "nunca",
                "Evaluación final": evaluaciones[-1][1],
            })
    display(pd.DataFrame(filas).style.hide(axis="index").format(precision=1, decimal=",", thousands="."))

## 3. ¿Ayuda `reward_slices`?

Los cuatro runs con **2 obstáculos verdes**: sin `reward_slices` (verde agua) y con `reward_slices` (azul), en 5×5 a la izquierda y en 3×3 a la derecha.

Cada 25.000 pasos el actor juega cinco partidas eligiendo siempre su acción más probable. El gráfico muestra el promedio de galletas por partida.

En la tabla, *galletas comidas al entrenar* es el total de galletas en las partidas con que se entrenó el modelo: sin galletas en los datos, no hay nada que aprender. *Primera evaluación con galletas* no cuenta la del paso 0, que es antes de entrenar.

In [ ]:
PREGUNTA_1 = [
    ("verdes, sin reward_slices", "Sin reward_slices"),
    ("verdes, con reward_slices", "Con reward_slices"),
]
graficar("¿Ayuda reward_slices? · 2 obstáculos verdes", PREGUNTA_1)
tabla(PREGUNTA_1)

## 4. ¿Importa cómo son los obstáculos?

Todos con `reward_slices`, porque los 3 obstáculos de colores solo se corrieron así: **2 obstáculos verdes** (azul) y **3 obstáculos de colores** (naranja). La línea gris punteada es el run **sin obstáculos** de la semana del 15, con la misma configuración. En 3×3 no hay run sin obstáculos con `reward_slices`.

In [ ]:
PREGUNTA_2 = [
    ("verdes, con reward_slices", "2 obstáculos verdes"),
    ("colores, con reward_slices", "3 obstáculos de colores"),
    ("sin obstáculos, con reward_slices", "Sin obstáculos (semana del 15)"),
]
graficar("¿Importa cómo son los obstáculos? · todos con reward_slices", PREGUNTA_2)
tabla(PREGUNTA_2)

## 5. Crear los videos que faltan (CPU)

Crea el video de la partida de semilla 0 para cada experimento de esta semana que no lo tenga o cuyo video sea anterior al último checkpoint. Juega el **actor aprendido** del último checkpoint, igual que en las evaluaciones.

Descarga una revisión fija del repositorio, instala las dependencias con PyTorch para CPU y usa el entorno de obstáculos de la semana de cada run. El video queda en la carpeta del run como `agente_episodio0.mp4`, el mismo nombre que usa la sección 7 de los notebooks de entrenamiento. Tarda unos minutos la primera vez y menos de un minuto por video.

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile

TRABAJO = Path("/content")
REPO = TRABAJO / "r2dreamer-videos"
URL = "https://github.com/FabriRandon/r2dreamer-cookie.git"
REVISION = "bdffa746f32dac6e6d9c7f37fc8f6138a671e117"
SEMILLA_VIDEO = 0
# Cada semana entrenó con su propia versión del entorno de obstáculos.
ENTORNO = {
    "semana24": "notebooks/experimentos-semana-del-24/cookie.py",
    "semana25": "notebooks/experimentos-semana-del-25/cookie.py",
}
# Los seis experimentos de esta semana, en el orden de los videos.
VIDEOS = [
    (vista, condicion)
    for vista in ("5×5", "3×3")
    for condicion in ("verdes, sin reward_slices", "verdes, con reward_slices", "colores, con reward_slices")
]


def video_de(carpeta):
    archivo = RAIZ / carpeta / f"agente_episodio{SEMILLA_VIDEO}.mp4"
    try:
        return archivo if archivo.is_file() and archivo.stat().st_size > 0 else None
    except OSError:
        return None  # Drive puede estar sincronizando el archivo


def al_dia(carpeta):
    """El video está al día si se creó después del último checkpoint."""
    archivo, checkpoint = video_de(carpeta), RAIZ / carpeta / "latest.pt"
    try:
        return archivo is not None and archivo.stat().st_mtime >= checkpoint.stat().st_mtime
    except OSError:
        return False


pendientes = []
for vista, condicion in VIDEOS:
    carpeta = RUNS[vista, condicion]
    if not (RAIZ / carpeta / "latest.pt").exists():
        print(f"{vista} · {condicion}: no hay checkpoint en Drive.")
    elif not al_dia(carpeta):
        pendientes.append((vista, condicion))

if not pendientes:
    print("Los seis videos ya están hechos con el último checkpoint.")
else:
    print("Falta 1 video." if len(pendientes) == 1 else f"Faltan {len(pendientes)} videos.", "Preparando el código…")
    if not (REPO / ".git").is_dir():
        REPO.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "init", "--quiet"], cwd=REPO, check=True)
        subprocess.run(["git", "fetch", "--quiet", "--depth=1", URL, REVISION], cwd=REPO, check=True)
        subprocess.run(["git", "checkout", "--quiet", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
    PYTHON = REPO / ".venv/bin/python"
    if not PYTHON.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        UV = [sys.executable, "-m", "uv"]
        subprocess.run([*UV, "venv", "--quiet", "--python", "3.11", ".venv"], cwd=REPO, check=True)
        (REPO / "cpu.txt").write_text("torch==2.8.0+cpu\n")
        subprocess.run([*UV, "pip", "install", "--quiet", "--python", str(PYTHON),
                        "--extra-index-url", "https://download.pytorch.org/whl/cpu",
                        "--index-strategy", "unsafe-best-match", "--constraint", "cpu.txt",
                        "-e", ".[cookie]"], cwd=REPO, check=True)
    temporal = Path(tempfile.mkdtemp(prefix="videos-", dir=TRABAJO))
    for vista, condicion in pendientes:
        carpeta = RUNS[vista, condicion]
        print(f"Creando video: {vista} · {condicion}")
        shutil.copyfile(REPO / ENTORNO[carpeta.split("/")[0]], REPO / "envs/cookie.py")
        provisional = temporal / f"{carpeta.replace('/', '_')}.mp4"
        proceso = subprocess.run([str(PYTHON), "watch.py", "--logdir", str(RAIZ / carpeta),
                                  "--out", str(provisional), "--seed", str(SEMILLA_VIDEO)],
                                 cwd=REPO, env={**os.environ, "CUDA_VISIBLE_DEVICES": ""},
                                 text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if proceso.returncode == 0 and provisional.exists():
            shutil.copyfile(provisional, RAIZ / carpeta / f"agente_episodio{SEMILLA_VIDEO}.mp4")
            print("  ", proceso.stdout.strip().splitlines()[-1].split(" Video en ")[0])
        else:
            print("   No se pudo crear. Vuelve a ejecutar esta celda.")
            print("\n".join(proceso.stdout.splitlines()[-10:]))

## 6. Ver los videos

Los seis experimentos de esta semana, por vista. En cada video, a la izquierda está el mapa completo y a la derecha lo que recibe el agente. El encabezado indica las galletas y los pasos entrenados del checkpoint.

En el 3×3 con obstáculos verdes y `reward_slices`, el último checkpoint guardado en Drive es de 370.000 pasos, aunque sus métricas llegan a 500.000. Su evaluación fue 0 durante todo el run.

In [ ]:
import html
from IPython.display import HTML, Video

TITULOS = {
    "verdes, sin reward_slices": ("2 obstáculos verdes", "sin reward_slices"),
    "verdes, con reward_slices": ("2 obstáculos verdes", "con reward_slices"),
    "colores, con reward_slices": ("3 obstáculos de colores", "con reward_slices"),
}


def tarjeta(vista, condicion):
    carpeta = RUNS[vista, condicion]
    obstaculos, slices = TITULOS[condicion]
    titulo = f"<h4>{html.escape(obstaculos)}</h4><p>{html.escape(slices)}</p>"
    archivo = video_de(carpeta)
    if archivo is None:
        return f'<div class="tarjeta">{titulo}<p>No hay video. Ejecuta la sección 5.</p></div>'
    try:
        reproductor = Video(str(archivo), embed=True)._repr_html_()
    except OSError:
        return f'<div class="tarjeta">{titulo}<p>Drive aún no pudo leer el video. Vuelve a ejecutar esta celda.</p></div>'
    nota = ""
    if (vista, condicion) in DATOS:
        final = DATOS[vista, condicion][0][-1][1]
        nota = f"Evaluación final: {final:.1f} galletas por partida.".replace(".", ",", 1)
    if not al_dia(carpeta):
        nota += " ⚠️ Video anterior al último checkpoint: ejecuta la sección 5."
    return f'<div class="tarjeta">{titulo}{reproductor}<p><small>{html.escape(nota)}</small></p></div>'


estilos = (
    "<style>"
    ".vista{max-width:1200px;margin:0 0 28px}"
    ".vista h3{margin:0 0 10px}"
    ".fila{display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:16px}"
    ".tarjeta{border:1px solid #8885;border-radius:8px;padding:12px}"
    ".tarjeta h4{margin:0}"
    ".tarjeta p{margin:2px 0 8px}"
    ".tarjeta video{width:100%;height:auto}"
    "</style>"
)
bloques = []
for vista in ("5×5", "3×3"):
    tarjetas = "".join(tarjeta(v, condicion) for v, condicion in VIDEOS if v == vista)
    bloques.append(f'<div class="vista"><h3>Vista {vista}</h3><div class="fila">{tarjetas}</div></div>')
display(HTML(estilos + "".join(bloques)))